In [1]:
import gc, gzip, pickle, xgboost, shap, numpy as np

import polars as pl 
_=pl.Config.set_tbl_cols(100000)
_=pl.Config.set_tbl_rows(10000)
_=pl.Config.set_tbl_width_chars(10000)
_=pl.Config.set_fmt_str_lengths(10000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

from loguru import logger
from pathlib import Path
from sklearn.metrics import r2_score

In [ ]:
MASTER_DICT = [
    {
        "gamma": 0.5,
        "n_jobs": -1,
        "max_depth": 3,
        "objective": "reg:logistic",
        "reg_alpha": 0.5,
        "subsample": 1.0,
        "reg_lambda": 0.1,
        "eval_metric": ["rmse","logloss"],
        "n_estimators": 100000,
        "learning_rate": 0.05,
        "colsample_bytree": 0.75,
        "min_child_weight": 30,
        "early_stopping_rounds": 100,
        "cell_line": "HepG2",
    },
    {
        "gamma": 0.5,
        "n_jobs": -1,
        "max_depth": 5,
        "objective": "reg:logistic",
        "reg_alpha": 0.8,
        "subsample": 1.0,
        "reg_lambda": 10,
        "eval_metric": ["rmse", "logloss"],
        "n_estimators": 100000,
        "learning_rate": 0.05,
        "colsample_bytree": 0.5,
        "min_child_weight": 1,
        "early_stopping_rounds": 100,
        "cell_line": "K562",
    }
]

SEED = 17
DATA_DIR = "/project/PlatigLab/data/RBP_ML/3_yogi_dataset_feb_2025"
DATA_SPLITTING_INDICES_DIR = "/project/PlatigLab/users/yogi/ENCODE-RNA-Binding-Protein-Network-Modeling/analysis/03_choose_dataset_and_model_parameters/output/model_reproduction/rows_to_reproduce"



####################################################
### LOAD DATA & FILTER & DO IN-SILICO KNOCKDOWNS ###
####################################################


In [ ]:
for dict in MASTER_DICT:
    cell_line = dict["cell_line"]

    logger.info(f"Processing cell line: {cell_line}")

    file_pattern = f"{cell_line}_100_all-events_num-peaks-no-kd.tsv.gz"
    file_path = Path(DATA_DIR) / file_pattern
    chromosomes_allowed = [f"chr{i}" for i in range(1, 23)]

    logger.info(f"Loading data from: {file_path}")

    df = pl.scan_csv(file_path, separator="\t")

    df = (
        df
        .filter(
            (pl.col("Total Read Counts") >= 40)
            & (pl.col("chr").is_in(chromosomes_allowed))
        )
        .collect()
    ) 
    assert df["index"].n_unique() == df.shape[0], "The 'index' column contains duplicate values"

    binding_cols = [col for col in df.columns if col.endswith("_binding")]
    df = df.with_columns(
        [pl.when(pl.col(col) > 1).then(1).otherwise(pl.col(col)).alias(col) for col in binding_cols]
    )
    assert all(df[col].max() <= 1 for col in binding_cols), "Some values in binding columns are greater than 1"

    original_df_shape = df.shape
    logger.info(f"Original dataframe shape: {original_df_shape}")

    unique_rbp_kd_targets = sorted(df["RBP_KD_Target"].unique().to_list())
    modified_dfs = []
    for rbp_kd_target in unique_rbp_kd_targets:
        subset_df = df.filter(pl.col("RBP_KD_Target") == rbp_kd_target)

        if rbp_kd_target != "CTRL":
            binding_cols_to_zero = [col for col in binding_cols if col.startswith(f"{rbp_kd_target}_")]
            assert len(binding_cols_to_zero) ==6, print(binding_cols_to_zero)
            
            subset_df = subset_df.with_columns([
                pl.lit(0).alias(col) for col in binding_cols_to_zero
            ])

        modified_dfs.append(subset_df)

    df = pl.concat(modified_dfs, how='vertical_relaxed')

    del subset_df, modified_dfs
    gc.collect()

    assert df.shape == original_df_shape, "Dataframe shape changed after modification"
    df = df.sort("index").select(
        [col for col in df.columns if col.endswith("_binding") or col in ["Target_PSI", 'index']]
    )

    df.write_ipc(f'{cell_line}.feather')
    logger.info(f"Data saved to {cell_line}.feather")

    del df
    gc.collect()

##################################################
### LOAD DATA & SPLIT INTO TRAIN/VALIDATE/TEST ###
##################################################

In [ ]:
for dict in MASTER_DICT:
    cell_line = dict["cell_line"]

    df = pl.scan_ipc(f'{cell_line}.feather')
    logger.info(f"Loaded data for cell line: {cell_line}")

    data_splitting_indices_path = Path(DATA_SPLITTING_INDICES_DIR) / f"{cell_line}.pkl.gz"
    logger.info(f"Loading data splitting indices from: {data_splitting_indices_path}")

    with gzip.open(data_splitting_indices_path, 'rb') as f:
        indices_dict = pickle.load(f)

    brett_metadata_df = indices_dict["metadata_df"]

    for split in ["train", "validate", "test"]:
        indices = indices_dict[f"{split}_meta_ind"]

        split_indices = brett_metadata_df[
            brett_metadata_df['index'].isin(indices)
        ]["unique_id"].to_list()

        assert len(split_indices) == len(indices), f"Length mismatch for {split}: {len(split_indices)} vs {len(indices)}"

        tmp_df = df.filter(pl.col("index").is_in(split_indices)).drop('index').collect()

        tmp_df.write_ipc(f'{cell_line}_{split}.feather')
        logger.info(f"Data for {split} split saved to {cell_line}_{split}.feather")


###########################
### TRAIN XGBOOST MODEL ###
###########################


In [ ]:

for params in MASTER_DICT:
    cell_line = params["cell_line"]
    logger.info(f"Training XGBoost model for cell line: {cell_line}")

    # Prepare model parameters (exclude 'cell_line')
    model_params = {k: v for k, v in params.items() if k != "cell_line"}

    # Load train and validate data as lazyframes
    train_lf = pl.scan_ipc(f"{cell_line}_train.feather")
    validate_lf = pl.scan_ipc(f"{cell_line}_validate.feather")

    # Collect only the columns needed for training to reduce memory usage
    binding_cols = [col for col in train_lf.collect_schema().names() if col.endswith("_binding")]
    X_train = train_lf.select(binding_cols).collect().to_pandas()
    y_train = train_lf.select(["Target_PSI"]).collect().to_pandas()["Target_PSI"]
    X_valid = validate_lf.select(binding_cols).collect().to_pandas()
    y_valid = validate_lf.select(["Target_PSI"]).collect().to_pandas()["Target_PSI"]

    model = xgboost.XGBRegressor(**model_params, random_state=SEED)
    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=False,
    )

    logger.info(f"Model training complete for cell line: {cell_line}")

    # Save the trained model
    model.save_model(f"{cell_line}_xgb_model.json")
    logger.info(f"Model saved to {cell_line}_xgb_model.json")

    # Load test data
    test_lf = pl.scan_ipc(f"{cell_line}_test.feather")
    X_test = test_lf.select(binding_cols).collect().to_pandas()
    y_test = test_lf.select(["Target_PSI"]).collect().to_pandas()["Target_PSI"]

    # Predict and compute R2 score
    y_pred = model.predict(X_test)
    test_r2 = r2_score(y_test, y_pred)
    logger.info(f"Test R2 score for {cell_line}: {test_r2:.4f}")
    
    del X_train, y_train, X_valid, y_valid, X_test, y_test, model
    gc.collect()



############################
### RETRIEVE SHAP VALUES ###
############################


In [3]:

for params in MASTER_DICT[1:]:
    cell_line = params["cell_line"]
    logger.info(f"Computing SHAP values for cell line: {cell_line}")

#     # Load model
#     model = xgboost.XGBRegressor()
#     model.load_model(f"{cell_line}_xgb_model.json")

    with gzip.open("../../04_run_final_models_and_SHAP/outputs/pickled_models/XGBRegressor/f93fa76e4f7b67d2494c9ef5eadbafe3f38bb4f2cad488040116e580f0cc0de2.pkl.gz", 'rb') as f:
        model = pickle.load(f)

    # Load train and validate data
    train_lf = pl.scan_ipc(f"{cell_line}_train.feather")
    validate_lf = pl.scan_ipc(f"{cell_line}_validate.feather")
    binding_cols = [col for col in train_lf.collect_schema().names() if col.endswith("_binding")]

    X_train = train_lf.select(binding_cols).collect().to_pandas()
    X_valid = validate_lf.select(binding_cols).collect().to_pandas()
    X_concat = pd.concat([X_train, X_valid], ignore_index=True)

    del X_train, X_valid
    gc.collect()

    # TreeExplainer with interventional and probability output
    explainer = shap.TreeExplainer(model, data = X_concat, feature_perturbation="interventional", model_output="probability")

    # Concatenate test data and compute SHAP values for all data
    test_lf = pl.scan_ipc(f"{cell_line}_test.feather")
    all_data = test_lf.select(binding_cols).collect().to_pandas()
    all_data = pd.concat([X_concat, all_data], ignore_index=True)

    for i in range(100):
        sample = all_data.sample(n=100, random_state=SEED + i)
    
        shap_values = explainer.shap_values(sample)
        shap_sum = shap_values.sum(axis=1)
        expected_value = explainer.expected_value
        shap_pred = shap_sum + expected_value

        model_pred = model.predict(sample)

        diff = model_pred - shap_pred
        
        print(f"Sample {i+1}: mean(abs(diff))={np.mean(np.abs(diff)):.6f}, max(abs(diff))={np.max(np.abs(diff)):.6f}")

    del X_concat, explainer, model
    gc.collect()



2025-08-02 16:42:39.688 | INFO     | __main__:<module>:3 - Computing SHAP values for cell line: K562


0

Sample 1: mean(abs(diff))=0.000000, max(abs(diff))=0.000000
Sample 2: mean(abs(diff))=0.000000, max(abs(diff))=0.000000
Sample 3: mean(abs(diff))=0.000000, max(abs(diff))=0.000000
Sample 4: mean(abs(diff))=0.000000, max(abs(diff))=0.000000
Sample 5: mean(abs(diff))=0.000000, max(abs(diff))=0.000000
Sample 6: mean(abs(diff))=0.000000, max(abs(diff))=0.000000
Sample 7: mean(abs(diff))=0.000000, max(abs(diff))=0.000000
Sample 8: mean(abs(diff))=0.000000, max(abs(diff))=0.000000
Sample 9: mean(abs(diff))=0.000000, max(abs(diff))=0.000000
Sample 10: mean(abs(diff))=0.000000, max(abs(diff))=0.000000
Sample 11: mean(abs(diff))=0.000000, max(abs(diff))=0.000000
Sample 12: mean(abs(diff))=0.000000, max(abs(diff))=0.000000
Sample 13: mean(abs(diff))=0.000000, max(abs(diff))=0.000000
Sample 14: mean(abs(diff))=0.000000, max(abs(diff))=0.000000
Sample 15: mean(abs(diff))=0.000000, max(abs(diff))=0.000000
Sample 16: mean(abs(diff))=0.000000, max(abs(diff))=0.000000
Sample 17: mean(abs(diff))=0.0000

27

In [ ]:

for params in MASTER_DICT[1:]:
    cell_line = params["cell_line"]
    logger.info(f"Computing SHAP values for cell line: {cell_line}")

    with gzip.open("../../04_run_final_models_and_SHAP/outputs/pickled_models/XGBRegressor/f93fa76e4f7b67d2494c9ef5eadbafe3f38bb4f2cad488040116e580f0cc0de2.pkl.gz", 'rb') as f:
        model = pickle.load(f)

    lazy_k562 = pd.read_feather("../../04_run_final_models_and_SHAP/outputs/predictions/XGBRegressor/f93fa76e4f7b67d2494c9ef5eadbafe3f38bb4f2cad488040116e580f0cc0de2.feather")
    binding_input = lazy_k562[lazy_k562["Partition"].isin(["Train", "Validate"])][[col for col in lazy_k562.columns if col.endswith("_binding")]]
    
    assert list(binding_input.columns) == list(model.column_order_when_fitting), "Column order mismatch between input data and model."
    binding_input = binding_input[model.column_order_when_fitting]
    assert list(binding_input.columns) == list(model.column_order_when_fitting), "Column order mismatch between input data and model."
            
    # TreeExplainer with interventional and probability output
    explainer = shap.TreeExplainer(model, data = binding_input, feature_perturbation="interventional", model_output="probability")

    for i in range(100):
        sample = binding_input.sample(n=100, random_state=SEED + i)
    
        shap_values = explainer.shap_values(sample)
        shap_sum = shap_values.sum(axis=1)
        expected_value = explainer.expected_value
        shap_pred = shap_sum + expected_value
        
        shap_df = pd.DataFrame(shap_values)
        result = pd.concat([sample, shap_df], axis=1, ignore_index=True)


        model_pred = model.predict(sample)

        diff = model_pred - shap_pred
        
        print(f"Sample {i+1}: mean(abs(diff))={np.mean(np.abs(diff)):.7f}, max(abs(diff))={np.max(np.abs(diff)):.7f}")
        
        shap_values
        result
        sys.exit(0)
    
#     del X_concat, explainer, model
#     gc.collect()

